## Importing Dependencies

In [1]:
from datasets import load_dataset
import pandas as pd

## Fetching Queries from the gretelai dataset

In [2]:
# Fetching the gretelai dataset
dataset = load_dataset("gretelai/synthetic_text_to_sql", split="train")
# Converting the raw dataset into a dataframe 
dataframe = dataset.to_pandas()

print(dataframe.columns)
print(dataframe.head(2))

Index(['id', 'domain', 'domain_description', 'sql_complexity',
       'sql_complexity_description', 'sql_task_type',
       'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql',
       'sql_explanation'],
      dtype='str')
     id            domain                                 domain_description  \
0  5097          forestry  Comprehensive data on sustainable forest manag...   
1  5098  defense industry  Defense contract data, military equipment main...   

  sql_complexity                         sql_complexity_description  \
0    single join        only one join (specify inner, outer, cross)   
1    aggregation  aggregation functions (COUNT, SUM, AVG, MIN, M...   

             sql_task_type                          sql_task_type_description  \
0  analytics and reporting  generating reports, dashboards, and analytical...   
1  analytics and reporting  generating reports, dashboards, and analytical...   

                                          sql_prompt  \
0  What is

In [3]:
# Finding total number of different types of complexities present in the dataset
print(dataframe['sql_complexity'].unique())

# keep only SELECT queries
dataframe = dataframe[
    dataframe['sql'].str.strip().str.upper().str.startswith('SELECT')
]

# Seperating out different types of labels according to their description
simple_labels = ['basic SQL']
moderate_labels = ['aggregation', 'subqueries', 'single join']
complex_labels = ['multiple_joins', 'window functions', 'set operations', 'CTEs']

total_simple_quries = dataframe['sql_complexity'].isin(simple_labels).sum()
total_moderate_queries = dataframe['sql_complexity'].isin(moderate_labels).sum()
total_complex_queries = dataframe['sql_complexity'].isin(complex_labels).sum()

print(f"Simple queries: {total_simple_quries}")
print(f"Moderate queries: {total_moderate_queries}")
print(f"Complex queries: {total_complex_queries}")

# Finding the splits for the simple, moderate and complex splits
simple_counts = 30
moderate_counts = 40
complex_counts = 50

simple_queries = dataframe[dataframe['sql_complexity'].isin(simple_labels)].sample(n=simple_counts)
moderate_queries = dataframe[dataframe['sql_complexity'].isin(moderate_labels)].sample(n=moderate_counts)
complex_queries = dataframe[dataframe['sql_complexity'].isin(complex_labels)].sample(n=complex_counts)

output_dataframe = pd.concat([simple_queries, moderate_queries, complex_queries]).sample(frac=1).reset_index(drop=True)
print(output_dataframe.head(5))
print(dataframe['sql_complexity'].unique())
output_dataframe.to_csv("/Users/devrathod/Documents/classes_2026/ECS_189G/final_project/SQLChange/data/queries.csv", index=False)

<ArrowStringArray>
[     'single join',      'aggregation',        'basic SQL',
 'window functions',       'subqueries',   'multiple_joins',
   'set operations',             'CTEs']
Length: 8, dtype: str
Simple queries: 39304
Moderate queries: 42693
Complex queries: 7487
      id              domain  \
0  32344              retail   
1  28477   rural development   
2  36856  financial services   
3  73506  defense operations   
4  47617       public health   

                                  domain_description    sql_complexity  \
0  Retail data on circular supply chains, ethical...    multiple_joins   
1  Agricultural innovation metrics, rural infrast...         basic SQL   
2  Detailed financial data including investment s...  window functions   
3  Defense data on military innovation, peacekeep...  window functions   
4  Community health statistics, infectious diseas...       aggregation   

                          sql_complexity_description            sql_task_type  \
0    two 

In [4]:
import sqlglot
from sqlglot import exp
import pandas as pd

df = pd.read_csv("/Users/devrathod/Documents/classes_2026/ECS_189G/final_project/SQLChange/data/queries.csv")

has_where    = 0
has_join     = 0
has_group_by = 0
has_limit    = 0
has_agg      = 0

for sql in df['sql']:
    try:
        tree = sqlglot.parse_one(sql)
        if tree.find(exp.Where):   has_where    += 1
        if tree.find(exp.Join):    has_join      += 1
        if tree.find(exp.Group):   has_group_by  += 1
        if tree.find(exp.Limit):   has_limit     += 1
        if tree.find(exp.AggFunc): has_agg       += 1
    except:
        pass

print(f"Has WHERE:    {has_where}/110")
print(f"Has JOIN:     {has_join}/110")
print(f"Has GROUP BY: {has_group_by}/110")
print(f"Has LIMIT:    {has_limit}/110")
print(f"Has AGG:      {has_agg}/110")

Has WHERE:    88/110
Has JOIN:     34/110
Has GROUP BY: 50/110
Has LIMIT:    3/110
Has AGG:      97/110
